<a href="https://colab.research.google.com/github/darixbp/03MIAR-Algoritmos-de-Optimizacion/blob/main/Ejemplo_Kruskal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
"""
Implementación del algoritmo de Kruskal para el diseño de red FTTH interurbana.
Ejemplo: Conexión óptima entre sedes (Madrid, Barcelona, Valencia, Sevilla).
"""

class DisjointSetCity:
    """
    Estructura Union-Find con compresión de caminos y unión por rango.
    Permite gestionar los grupos de ciudades y detectar ciclos.
    """
    def __init__(self, cities):
        # Fase 1: Cada sede empieza en su propio conjunto independiente
        self.parent = {city: city for city in cities}
        self.rank = {city: 0 for city in cities}

    def find(self, city):
        # Compresión de caminos: apunta directamente a la raíz del grupo
        if self.parent[city] != city:
            self.parent[city] = self.find(self.parent[city])
        return self.parent[city]

    def union(self, city1, city2):
        root1 = self.find(city1)
        root2 = self.find(city2)

        # Si ya comparten raíz, pertenecen al mismo grupo (crearían un bucle)
        if root1 == root2:
            return False

        # Unión por rango para mantener el árbol balanceado
        if self.rank[root1] < self.rank[root2]:
            self.parent[root1] = root2
        elif self.rank[root1] > self.rank[root2]:
            self.parent[root2] = root1
        else:
            self.parent[root2] = root1
            self.rank[root1] += 1

        return True


def kruskal_sedes(ciudades, conexiones):
    print("=" * 60)
    print("EJECUCIÓN DEL ALGORITMO DE KRUSKAL")
    print("=" * 60)

    # --- FASE 1: Inicialización de los nodos ---
    print("\n[FASE 1] Inicializando sedes en grupos independientes")
    ds = DisjointSetCity(ciudades)
    print(f"Sedes registradas: {ciudades}\n")

    # --- FASE 2: Ordenación de las aristas ---
    # Formato de conexión: (coste, sede_origen, sede_destino)
    print("[FASE 2] Ordenando ofertas de cable por coste ascendente")
    conexiones_ordenadas = sorted(conexiones, key=lambda x: x[0])
    for coste, u, v in conexiones_ordenadas:
        print(f"  • {u} - {v}: {coste} €")
    print()

    # --- FASE 3: Comprobación y detección de ciclos ---
    print("[FASE 3] Evaluando conexiones para evitar redundancias")
    mst = []
    coste_total = 0

    for coste, u, v in conexiones_ordenadas:
        print(f"\nEvaluando enlace {u} <--> {v} ({coste} €):")

        # Intentamos unir los conjuntos de ambas ciudades
        if ds.union(u, v):
            mst.append((u, v, coste))
            coste_total += coste
            print(f"  -> ACEPTADO: No existía ruta previa. Se adquiere el cable.")
        else:
            print(f"  -> RECHAZADO: {u} y {v} ya están interconectadas. Evitamos ciclo.")

        # Condición de parada temprana: el MST se completa con (n - 1) aristas
        if len(mst) == len(ciudades) - 1:
            print("\n[INFO] Se han alcanzado n - 1 conexiones. Todas las sedes están unidas.")
            break

    return mst, coste_total


# -------------------------------------------------------------------------
# EJEMPLO PLANTEADO EN LA PRIMERA ENTRADA DEL FORO
# -------------------------------------------------------------------------
if __name__ == "__main__":
    sedes = ["Madrid", "Barcelona", "Valencia", "Sevilla"]

    # Conexiones disponibles y sus costes asociados (a)
    ofertas_fibra = [
        (400, "Madrid", "Sevilla"),
        (100, "Madrid", "Barcelona"),
        (50, "Barcelona", "Valencia"),
        (200, "Valencia", "Sevilla")
    ]

    arbol_minimo, coste_final = kruskal_sedes(sedes, ofertas_fibra)

    # ---------------------------------------------------------------------
    # RESULTADO FINAL
    # ---------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("RED FINAL SELECCIONADA (ÁRBOL DE RECUBRIMIENTO MÍNIMO)")
    print("=" * 60)
    for u, v, coste in arbol_minimo:
        print(f"  • {u} <---> {v} | Coste: {coste} €")

    print("-" * 60)
    print(f"Inversión total óptima: {coste_final} €")
    print("=" * 60)

EJECUCIÓN DEL ALGORITMO DE KRUSKAL

[FASE 1] Inicializando sedes en grupos independientes
Sedes registradas: ['Madrid', 'Barcelona', 'Valencia', 'Sevilla']

[FASE 2] Ordenando ofertas de cable por coste ascendente
  • Barcelona - Valencia: 50 €
  • Madrid - Barcelona: 100 €
  • Valencia - Sevilla: 200 €
  • Madrid - Sevilla: 400 €

[FASE 3] Evaluando conexiones para evitar redundancias

Evaluando enlace Barcelona <--> Valencia (50 €):
  -> ACEPTADO: No existía ruta previa. Se adquiere el cable.

Evaluando enlace Madrid <--> Barcelona (100 €):
  -> ACEPTADO: No existía ruta previa. Se adquiere el cable.

Evaluando enlace Valencia <--> Sevilla (200 €):
  -> ACEPTADO: No existía ruta previa. Se adquiere el cable.

[INFO] Se han alcanzado n - 1 conexiones. Todas las sedes están unidas.

RED FINAL SELECCIONADA (ÁRBOL DE RECUBRIMIENTO MÍNIMO)
  • Barcelona <---> Valencia | Coste: 50 €
  • Madrid <---> Barcelona | Coste: 100 €
  • Valencia <---> Sevilla | Coste: 200 €
------------------------